# <a id='toc1_'></a>[Title](#toc0_)
- Member 1
- Member 2
- Member 3



**Table of contents**<a id='toc0_'></a>    
  - [Import libraries and data](#toc1_1_)    
  - [Part 1: Preprocessing the documents](#toc1_2_)    
  - [Part 2: EDA](#toc1_3_)    

<!-- vscode-jupyter-toc-config
	numbering=false
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

## <a id='toc1_1_'></a>[Import libraries and data](#toc0_)

In [14]:
import pandas as pd
from nltk.stem import PorterStemmer
from nltk.tokenize import RegexpTokenizer
from nltk.corpus import stopwords
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/mvelasco/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [15]:
# Read fashion products dataset
data_dir = "../../data/"
filename = "fashion_products_dataset.json"
df = pd.read_json(data_dir+filename)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28080 entries, 0 to 28079
Data columns (total 17 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   _id              28080 non-null  object        
 1   actual_price     28080 non-null  object        
 2   average_rating   28080 non-null  object        
 3   brand            28080 non-null  object        
 4   category         28080 non-null  object        
 5   crawled_at       28080 non-null  datetime64[ns]
 6   description      28080 non-null  object        
 7   discount         28080 non-null  object        
 8   images           28080 non-null  object        
 9   out_of_stock     28080 non-null  bool          
 10  pid              28080 non-null  object        
 11  product_details  28080 non-null  object        
 12  seller           28080 non-null  object        
 13  selling_price    28080 non-null  object        
 14  sub_category     28080 non-null  objec

## <a id='toc1_2_'></a>[Part 1: Preprocessing the documents](#toc0_)

In [16]:
def build_terms(text):
    """
    Preprocess text removing stop words, stemming,
    transforming in lowercase and return the tokens of the text.

    Argument:
    text -- string (text) to be preprocessed

    Returns:
    text - a list of tokens corresponding to the input text after the preprocessing
    """

    stemmer = PorterStemmer()
    tokenizer = RegexpTokenizer(r'\w+') # keep only words & numbers, drop punctuation
    stop_words = set(stopwords.words("english"))
    text= text.lower() # transform in lowercase
    text=  tokenizer.tokenize(text) # tokenize the text to get a list of terms
    text=[t for t in text if t not in stop_words]  # eliminate the stopwords
    text=[stemmer.stem(t) for t in text] # perform stemming
    return text

test_row = df.iloc[0]
print(f"Before: {test_row['description']}")
print(f"After: {build_terms(test_row['description'])}")

Before: Yorker trackpants made from 100% rich combed cotton giving it a rich look.Designed for Comfort,Skin friendly fabric,itch-free waistband & great for all year round use Proudly made in India
After: ['yorker', 'trackpant', 'made', '100', 'rich', 'comb', 'cotton', 'give', 'rich', 'look', 'design', 'comfort', 'skin', 'friendli', 'fabric', 'itch', 'free', 'waistband', 'great', 'year', 'round', 'use', 'proudli', 'made', 'india']


In [17]:
text_cols = ["title", "description", "brand", "category", "sub_category", "seller"]

def merge_text_cols(row, text_cols=text_cols):
    # Turn product detail values into a string (e.g. 'blue nylon checkered')
    details = row['product_details']
    details = {k: v for d in details for k, v in d.items()} # convert to dict
    details = list(details.values()) # keep only values 
    details = ' '.join(details)

    # Merges all text in text cols into a single string
    text = ' '.join(row[text_cols].fillna('').astype(str)) 

    return text + ' ' + details

print(f"Before:")
display(df[text_cols + ['product_details']].head(1))
print(f"After:")
merge_text_cols(test_row)

Before:


,title,description,brand,category,sub_category,seller,product_details
0,Solid Women Multicolor Track Pants,Yorker trackpants made from 100% rich combed c...,York,Clothing and Accessories,Bottomwear,Shyam Enterprises,"[{'Style Code': '1005COMBO2'}, {'Closure': 'El..."


After:


'Solid Women Multicolor Track Pants Yorker trackpants made from 100% rich combed cotton giving it a rich look.Designed for Comfort,Skin friendly fabric,itch-free waistband & great for all year round use Proudly made in India York Clothing and Accessories Bottomwear Shyam Enterprises 1005COMBO2 Elastic Side Pockets Cotton Blend Solid Multicolor'

In [ ]:
df['text'] = df.apply(merge_text_cols, axis=1) 
df['processed_text'] = df['text'].apply(build_terms) # Preprocess text (lowercase, stem, etc.)

df.head(3)

In [ ]:
outpath = data_dir + f"procesed_fashion_products.csv"
df.to_csv(outpath)
print(f"✅ Saved processed data to {outpath}")

✅ Saved processed data to ../../data/procesed_fashion_products_dataset.json


## <a id='toc1_3_'></a>[Part 2: EDA](#toc0_)